In [9]:
# Load MSK data (if not already loaded)
# Uncomment if needed:
import sys,os
# Add parent directory to path to import modules from one level up
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, parent_dir)
import importlib
import model_IAI
importlib.reload(model_IAI)
from model_IAI import *
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("DataLoad").getOrCreate()
# Load the MSK dataset with enhanced cost features
df_msk_spark = spark.read.format("parquet").load("msk_2017_18_full.parquet")
df_og = df_msk_spark.toPandas()

# MSK-specific feature column definitions
# Binary flag columns: comorbidity flags, MSK category flags, medication flags
BIN_FLAG_COLUMNS = get_bin_flag_columns(df_og)

# MSK doesn't have stage columns like CKD, but we can identify categorical cost pattern/stability columns
STAGE_COLUMNS = []  # MSK doesn't use stage columns

# Categorical columns
CAT_COLUMNS = get_cat_columns(df_og)
# True numeric columns (excluding binary flags and categorical)
TRUE_NUM_COLUMNS = get_true_num_columns(df_og, CAT_COLUMNS,BIN_FLAG_COLUMNS)


# Cost columns (2017 only - exclude 2018 to prevent leakage)
COST_COLUMNS = [
    col for col in df_og.columns 
    if ("cost" in col.lower() or "quarterly" in col.lower() or "increasing" in col.lower() or 
        "decreasing" in col.lower() or "skewness" in col.lower() or "kurtosis" in col.lower() or
        "cv" in col.lower() or "range" in col.lower())
    and "2018" not in col  # Exclude 2018 columns to prevent leakage
]

# Utilization columns (if any)
UTILIZATION_COLUMNS = [col for col in df_og.columns if "claims" in col.lower() and "2018" not in col]

print("categorical cols: ", CAT_COLUMNS[:10], f"... ({len(CAT_COLUMNS)} total)")
print("stage cols: ", STAGE_COLUMNS)
print("cost cols (sample): ", COST_COLUMNS[:10], f"... ({len(COST_COLUMNS)} total)")

leftover_cols = [
    c for c in df_og.columns 
    if c not in CAT_COLUMNS and c not in TRUE_NUM_COLUMNS and c not in STAGE_COLUMNS and c not in BIN_FLAG_COLUMNS 
    and c != "ENROLID"
]

print(f"Number of leftover columns: {len(leftover_cols)}")
if len(leftover_cols) > 0:
    print("Leftover columns:", leftover_cols[:20])

# Create cost stratum from 2018 target (if available)
# Check what target columns are available
target_candidates = [col for col in df_og.columns if "2018" in col and ("top" in col.lower() or "pct" in col.lower() or "cost" in col.lower())]
print(f"\nAvailable 2018 target columns: {target_candidates}")

# Use top_2_pct_cost_2018 as target if available, otherwise create from annual_cost_2018_deflated
if "top_2_pct_cost_2018" in df_og.columns:
    target_col = "top_2_pct_cost_2018"
    print(f"Using {target_col} as target column")
elif "annual_cost_2018_deflated" in df_og.columns:
    # Create binary target from annual_cost_2018_deflated
    # Use 98th percentile as threshold (top 2%)
    threshold = df_og["annual_cost_2018_deflated"].quantile(0.98)
    df_og["top_2_pct_cost_2018"] = (df_og["annual_cost_2018_deflated"] >= threshold).astype(int)
    target_col = "top_2_pct_cost_2018"
    print(f"Created {target_col} using threshold ${threshold:,.2f}")
else:
    raise ValueError("No 2018 target column found. Need either 'top_2_pct_cost_2018' or 'annual_cost_2018_deflated'")

# Exclude all columns containing "2018" from features (to prevent leakage)
# Also exclude ENROLID and target column
exclude_cols = ["ENROLID", target_col] + [col for col in df_og.columns if "2018" in col]

feature_cols = [c for c in df_og.columns if c not in exclude_cols]

print(f"\nTotal columns in dataset: {len(df_og.columns)}")
print(f"Columns excluded (leakage prevention): {len(exclude_cols)}")
print(f"Feature columns: {len(feature_cols)}")

# Check for high correlation with target (optional - can be slow for large datasets)
if len(feature_cols) < 500:  # Only do this for reasonable number of features
    numeric_cols = df_og[feature_cols + [target_col]].select_dtypes(include=["number"]).columns
    if len(numeric_cols) > 0:
        corrs = df_og[numeric_cols].corr()[target_col].abs().sort_values(ascending=False)
        # Columns to drop (very high correlation, likely leakage)
        high_corr_cols = corrs[corrs > 0.95].index.tolist()
        # Remove the target column itself, if present
        high_corr_cols = [col for col in high_corr_cols if col != target_col]
        # Final filtered feature set
        feature_cols = [col for col in feature_cols if col not in high_corr_cols]
        if len(high_corr_cols) > 0:
            print(f"High corr features dropped (corr > 0.95): {high_corr_cols}")

print(f"\nFinal feature count: {len(feature_cols)}")
print(f"Target column: {target_col}")
print(f"Target distribution:\n{df_og[target_col].value_counts()}")

categorical cols:  ['direct_msk_cost_pattern_2017', 'direct_msk_cost_stability_2017', 'msk_procedure_cost_pattern_2017', 'msk_procedure_cost_stability_2017', 'comorbidity_only_cost_pattern_2017', 'comorbidity_only_cost_stability_2017', 'AGEGRP', 'SEX', 'REGION', 'EESTATU'] ... (10 total)
stage cols:  []
cost cols (sample):  ['2017Q1_direct_msk_cost_3month', '2017Q1_msk_procedure_cost_3month', '2017Q1_comorbidity_only_cost_3month', '2017Q2_direct_msk_cost_3month', '2017Q2_msk_procedure_cost_3month', '2017Q2_comorbidity_only_cost_3month', '2017Q3_direct_msk_cost_3month', '2017Q3_msk_procedure_cost_3month', '2017Q3_comorbidity_only_cost_3month', '2017Q4_direct_msk_cost_3month'] ... (72 total)
Number of leftover columns: 0

Available 2018 target columns: ['annual_cost_2018_deflated', 'top_10_pct_cost_2018', 'top_5_pct_cost_2018', 'top_2_pct_cost_2018']
Using top_2_pct_cost_2018 as target column

Total columns in dataset: 99
Columns excluded (leakage prevention): 6
Feature columns: 94

Fina

In [3]:
TRAIN_TEST_SEED = 123

In [ ]:
# Split data into train/test/val (same as multiobjective_bilevel.ipynb)
# Use the target_col defined in cell 0
train_ids, test_ids, train_pd, test_pd = train_test_split_enrol(
    df_og,
    target_col=target_col,  # Use target_col from cell 0 (e.g., "top_2_pct_cost_2018")
    test_size=0.3,
    verbose=False,
    random_state=TRAIN_TEST_SEED
)
print(f"Train shape: {train_pd.shape}, Test shape: {test_pd.shape}")
print("Feature cols:", len(feature_cols))

val_ids, test_ids, val_pd, test_pd = train_test_split_enrol(
    test_pd, 
    target_col=target_col,
    test_size=0.5,
    verbose=False, 
    random_state=TRAIN_TEST_SEED
)
X_test = test_pd[feature_cols]
y_test = test_pd[target_col]
X_val = val_pd[feature_cols]
y_val = val_pd[target_col]

print(f"Train: {train_pd.shape}, Val: {val_pd.shape}, Test: {test_pd.shape}")
print(f"Train target distribution:\n{train_pd[target_col].value_counts()}")
print(f"Target column: {target_col}")

Train shape: (175777, 99), Test shape: (75333, 99)
Feature cols: 94
Train: (175777, 99), Val: (37666, 99), Test: (37667, 99)
Train target distribution:
top_2_pct_cost_2018
0    172381
1      3396
Name: count, dtype: int64
Target column: top_2_pct_cost_2018


In [10]:
# ============================================================================
# STEP 1: PRECOMPUTE DISTANCES (Minority-Majority and Majority-Majority)
# ============================================================================
import h5py
from sklearn.impute import SimpleImputer
import time

# Import distance computation functions
# Try to import from parent directory or current directory
try:
    from precompute_distances import (
        get_preprocessor, compute_distances_batched, 
        save_distances_hdf5, precompute_leaf_dnn_memmap
    )
except ImportError:
    # If not found, add parent directory to path
    parent_projects_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if parent_projects_dir not in sys.path:
        sys.path.insert(0, parent_projects_dir)
    from precompute_distances import (
        get_preprocessor, compute_distances_batched, 
        save_distances_hdf5, precompute_leaf_dnn_memmap
    )

# Separate minority (cases) and majority (controls)
cases = train_pd[train_pd[target_col] == 1].copy()
controls = train_pd[train_pd[target_col] == 0].copy()

print(f"\nDataset split:")
print(f"  Cases (minority): {len(cases):,}")
print(f"  Controls (majority): {len(controls):,}")
print(f"  Ratio: {len(controls)/len(cases):.2f}:1")

# Prepare feature columns for preprocessing
#TODO: try removing cost columns from exclude_cols and/or 2018 columns + target column
exclude_cols = COST_COLUMNS + [col for col in train_pd.columns if "2018" in col]
drop_cols = ['ENROLID', target_col] + exclude_cols
feature_cols_prep = [c for c in train_pd.columns if c not in drop_cols]

# Impute missing values
print(f"\nPreprocessing features...")
cases_prep = cases[feature_cols_prep].copy()
controls_prep = controls[feature_cols_prep].copy()
train_prep = pd.concat([cases_prep, controls_prep], axis=0, ignore_index=True)

# Create preprocessor (matching OCT preprocessing)
preprocessor = get_preprocessor(
    X=train_prep,
    cat_cols=CAT_COLUMNS,
    num_cols=TRUE_NUM_COLUMNS,
    binary_cols=BIN_FLAG_COLUMNS,
    verbose = True
)

# Transform both cases and controls
print(f"  Features: {len(feature_cols_prep)}")

X_minority = preprocessor.fit_transform(cases_prep)
X_majority = preprocessor.transform(controls_prep)

print(f"  Preprocessed shapes: Minority {X_minority.shape}, Majority {X_majority.shape}")

# Create output directory
DISTANCES_DIR = "./precomputed_distances_msk"
os.makedirs(DISTANCES_DIR, exist_ok=True)

# ============================================================================
# 1. PRECOMPUTE MINORITY-MAJORITY DISTANCES
# ============================================================================
print(f"\n{'='*80}")
print("1. COMPUTING MINORITY-MAJORITY DISTANCES")
print(f"{'='*80}")

PN_H5_PATH = os.path.join(DISTANCES_DIR, "distances_majority_minority.h5")

# Check if already computed
if os.path.exists(PN_H5_PATH):
    print(f"  ✓ Found existing file: {PN_H5_PATH}")
    with h5py.File(PN_H5_PATH, 'r') as f:
        existing_maj = set(f['majority_enrolids'][:])
        existing_min = set(f['minority_enrolids'][:])
        current_maj = set(controls['ENROLID'].values)
        current_min = set(cases['ENROLID'].values)
        
        if existing_maj == current_maj and existing_min == current_min:
            print(f"  ✓ Existing distances match current data - skipping computation")
        else:
            print(f"  ⚠️  Existing distances don't match - recomputing...")
            distances_pn = compute_distances_batched(
                X_majority, X_minority, 
                batch_size=1000, 
                dtype=np.float32
            )
            save_distances_hdf5(
                distances_pn,
                controls['ENROLID'].values.astype(np.int64),
                cases['ENROLID'].values.astype(np.int64),
                PN_H5_PATH
            )
else:
    print(f"  Computing distances (this may take a while)...")
    distances_pn = compute_distances_batched(
        X_majority, X_minority, 
        batch_size=1000, 
        dtype=np.float32
    )
    save_distances_hdf5(
        distances_pn,
        controls['ENROLID'].values.astype(np.int64),
        cases['ENROLID'].values.astype(np.int64),
        PN_H5_PATH
    )

# ============================================================================
# 2. PRECOMPUTE MAJORITY-MAJORITY DISTANCES (for k-center)
# ============================================================================
print(f"\n{'='*80}")
print("2. COMPUTING MAJORITY-MAJORITY DISTANCES (for k-center)")
print(f"{'='*80}")

DNN_OUT_DIR = os.path.join(DISTANCES_DIR, f"global_dnn_seed_{TRAIN_TEST_SEED}")
os.makedirs(DNN_OUT_DIR, exist_ok=True)

dnn_matrix_npy = os.path.join(DNN_OUT_DIR, "leaf_global_dnn_matrix.npy")
dnn_enrolids_npy = os.path.join(DNN_OUT_DIR, "leaf_global_dnn_enrolids.npy")

# Check if already computed
if os.path.exists(dnn_matrix_npy) and os.path.exists(dnn_enrolids_npy):
    existing_dnn_ids = np.load(dnn_enrolids_npy)
    existing_set = set(existing_dnn_ids.astype(int))
    current_set = set(controls['ENROLID'].values.astype(int))
    
    if existing_set == current_set:
        print(f"  ✓ Found existing majority-majority distances:")
        print(f"    {dnn_matrix_npy}")
        print(f"    {dnn_enrolids_npy}")
    else:
        print(f"  ⚠️  Existing DNN files have different ENROLIDs - recomputing...")
        dnn_matrix_npy, dnn_enrolids_npy = precompute_leaf_dnn_memmap(
            X_majority_leaf=X_majority,
            majority_enrolids_leaf=controls['ENROLID'].values.astype(np.int64),
            out_dir=DNN_OUT_DIR,
            leaf_id="global",
            batch_size=750,
        )
else:
    print(f"  Computing majority-majority distances (this may take a while)...")
    dnn_matrix_npy, dnn_enrolids_npy = precompute_leaf_dnn_memmap(
        X_majority_leaf=X_majority,
        majority_enrolids_leaf=controls['ENROLID'].values.astype(np.int64),
        out_dir=DNN_OUT_DIR,
        leaf_id="global",
        batch_size=750,
    )

print(f"\n✓ Distance precomputation complete!")
print(f"  Minority-Majority: {PN_H5_PATH}")
print(f"  Majority-Majority: {dnn_matrix_npy}")


Dataset split:
  Cases (minority): 3,396
  Controls (majority): 172,381
  Ratio: 50.76:1

Preprocessing features...
→ Building preprocessor w/ conditional imputation:
   • Cat: OHE on: ['AGEGRP', 'SEX', 'REGION', 'EESTATU']
   • Binary: passthrough (no scaling) on: ['has_Anemia', 'has_Atherosclerotic_Heart_Disease', 'has_Chronic_Pain', 'has_Depression_Anxiety', 'has_Gastroesophageal_Reflux_Disease', 'has_General_Health_Check', 'has_Hyperlipidemia', 'has_Hypertension', 'has_Long_term_Drug_Therapy', 'has_Obesity', 'has_Sleep_Apnea', 'has_Type_2_Diabetes', 'has_Vitamin_D_Deficiency', 'has_Arthropathies', 'has_Dorsopathies', 'has_Soft_tissue', 'has_Osteopathies', 'has_Other_MSK']
  Features: 22
  Preprocessed shapes: Minority (3396, 36), Majority (172381, 36)

1. COMPUTING MINORITY-MAJORITY DISTANCES
  ✓ Found existing file: ./precomputed_distances_msk/distances_majority_minority.h5


TypeError: argument must be an integer

In [7]:
# ============================================================================
# STEP 2: K-CENTER UNDERSAMPLING
# ============================================================================
# Based on kcenter_hyperparameter_search_global.py

# Import two_stage_kcenter_match
try:
    import two_stage_kcenter_match
    importlib.reload(two_stage_kcenter_match)
    from two_stage_kcenter_match import two_stage_kcenter_then_match
except ImportError:
    # Try adding parent directory to path
    parent_projects_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if parent_projects_dir not in sys.path:
        sys.path.insert(0, parent_projects_dir)
    import two_stage_kcenter_match
    importlib.reload(two_stage_kcenter_match)
    from two_stage_kcenter_match import two_stage_kcenter_then_match

# Configuration for k-center matching
MATCHING_RATIO = 1  # 1:1 matching (can be changed)
CASE_WEIGHTING = None  # Options: None, "boundary"
USE_ADAPTIVE_POOL = True  # Options: True, False
SEED_METHOD = "smart"  # Options: "smart", "centroid", "density", "random"

print("="*80)
print("K-CENTER UNDERSAMPLING FOR MSK DATASET")
print("="*80)
print(f"\nConfiguration:")
print(f"  Matching ratio: 1:{MATCHING_RATIO}")
print(f"  Case weighting: {CASE_WEIGHTING}")
print(f"  Adaptive pool: {USE_ADAPTIVE_POOL}")
print(f"  Seed method: {SEED_METHOD}")

# Determine candidate pool size M
n_cases = len(cases)
n_controls = len(controls)
M = n_controls // 2  # Use half of controls as candidate pool

print(f"\nDataset statistics:")
print(f"  Cases (minority): {n_cases:,}")
print(f"  Controls (majority): {n_controls:,}")
print(f"  Candidate pool size (M): {M:,} ({M/n_controls*100:.1f}% of controls)")

# Run two-stage k-center matching
print(f"\n{'='*80}")
print("RUNNING TWO-STAGE K-CENTER MATCHING")
print(f"{'='*80}\n")

matching_start_time = time.perf_counter()

try:
    matching_result = two_stage_kcenter_then_match(
        leaf_controls_enrolids=controls['ENROLID'].values.astype(np.int64),
        leaf_cases_enrolids=cases['ENROLID'].values.astype(np.int64),
        leaf_nn_matrix_npy=dnn_matrix_npy,
        leaf_nn_enrolids_npy=dnn_enrolids_npy,
        pn_h5_path=PN_H5_PATH,
        M=M,
        use_adaptive_pool=USE_ADAPTIVE_POOL,
        tau=None,  # Auto-compute from 95th percentile
        plateau_eps=0.01,
        force_nearest_per_case=True,
        force_topm=1,
        assignment_topk_start=None,  # Exact matching
        seed_method=SEED_METHOD,
        matching_ratio=MATCHING_RATIO,
        X_majority_leaf=X_majority,
        case_weighting=CASE_WEIGHTING,
    )
    
    matching_end_time = time.perf_counter()
    matching_time = matching_end_time - matching_start_time
    
    # Extract results
    selected_control_enrolids = matching_result["selected_control_enrolids"]
    all_match_costs = matching_result["match_costs"]
    candidate_majority_enrolids = matching_result["candidate_majority_enrolids"]
    case_to_control_map = matching_result["case_to_control_map"]
    
    print(f"\n✓ Matching complete!")
    print(f"  Cases matched: {n_cases:,}")
    print(f"  Total selected controls: {len(selected_control_enrolids):,}")
    print(f"  Unique selected controls: {len(set(selected_control_enrolids)):,}")
    print(f"  Mean matching cost: {all_match_costs.mean():.4f}")
    print(f"  Matching time: {matching_time:.2f}s")
    
    # Build undersampled dataset
    print(f"\n{'='*80}")
    print("BUILDING UNDERSAMPLED TRAINING DATASET")
    print(f"{'='*80}\n")
    
    # Collect all minority samples (keep ALL)
    all_minority = train_pd[train_pd[target_col] == 1].copy()
    print(f"✓ Collected all minority samples: {len(all_minority):,}")
    
    # Get unique majority samples from matching result
    unique_majority_enrolids = list(set(selected_control_enrolids))
    selected_majority = train_pd[
        (train_pd[target_col] == 0) & 
        (train_pd['ENROLID'].isin(unique_majority_enrolids))
    ].copy()
    
    print(f"✓ Collected selected majority samples: {len(selected_majority):,}")
    
    # Combine minority and majority
    undersampled_training_data = pd.concat([all_minority, selected_majority], axis=0, ignore_index=True)
    
    print(f"\n✓ Undersampled dataset created:")
    print(f"   Total samples: {len(undersampled_training_data):,}")
    print(f"   Minority: {len(all_minority):,}")
    print(f"   Majority: {len(selected_majority):,}")
    print(f"   Ratio (maj:min): {len(selected_majority)/len(all_minority):.2f}:1")
    print(f"\nClass distribution:")
    print(undersampled_training_data[target_col].value_counts().sort_index())
    
    # Save undersampled dataset
    RESULTS_DIR = "./two_stage_kcenter_results_msk"
    os.makedirs(RESULTS_DIR, exist_ok=True)
    
    config_name = f"cw_{CASE_WEIGHTING}_pool_{USE_ADAPTIVE_POOL}_seed_{SEED_METHOD}"
    undersample_path = os.path.join(RESULTS_DIR, f"undersampled_{config_name}.csv")
    undersampled_training_data.to_csv(undersample_path, index=False)
    print(f"\n✓ Saved undersampled dataset: {undersample_path}")
    
except Exception as e:
    print(f"\n✗ ERROR in k-center matching:")
    print(f"  {e}")
    import traceback
    traceback.print_exc()
    raise

K-CENTER UNDERSAMPLING FOR MSK DATASET

Configuration:
  Matching ratio: 1:1
  Case weighting: None
  Adaptive pool: True
  Seed method: smart

Dataset statistics:
  Cases (minority): 3,396
  Controls (majority): 172,381
  Candidate pool size (M): 86,190 (50.0% of controls)

RUNNING TWO-STAGE K-CENTER MATCHING

  Seed selection method: 'smart'
    Smart seed selected: index 102 (mean dist to cases: 2.8395)
  Auto-computed tau (95th percentile of best distances): 1.0000
  Adaptive pool stopped at 17239 candidates (max cost: 0.4032)

✓ Matching complete!
  Cases matched: 3,396
  Total selected controls: 3,396
  Unique selected controls: 3,396
  Mean matching cost: 0.4729
  Matching time: 78.94s

BUILDING UNDERSAMPLED TRAINING DATASET

✓ Collected all minority samples: 3,396
✓ Collected selected majority samples: 3,396

✓ Undersampled dataset created:
   Total samples: 6,792
   Minority: 3,396
   Majority: 3,396
   Ratio (maj:min): 1.00:1

Class distribution:
top_2_pct_cost_2018
0    3396

# Evaluate final OCT


In [8]:
import os, pickle
import model_IAI
importlib.reload(model_IAI)
from model_IAI import evaluate_binary_oct, finetune_oct
ratio_values = [1.0]

In [ ]:
import time
try:
    import psutil
    PSUTIL_AVAILABLE = True
except ImportError:
    PSUTIL_AVAILABLE = False
    print("Warning: psutil not available. Resource tracking will be limited.")

def format_time(seconds):
    """Format time in a readable way."""
    if seconds < 0.1:
        return f"{seconds:.4f}"
    elif seconds < 1.0:
        return f"{seconds:.3f}"
    else:
        return f"{seconds:.2f}"

def get_resource_usage():
    """Get current CPU and memory usage."""
    if PSUTIL_AVAILABLE:
        process = psutil.Process()
        memory_info = process.memory_info()
        return {
            'cpu_percent': process.cpu_percent(interval=0.1),
            'memory_mb': memory_info.rss / (1024 * 1024),  # RSS in MB
            'memory_percent': process.memory_percent(),
        }
    else:
        return {
            'cpu_percent': None,
            'memory_mb': None,
            'memory_percent': None,
        }

# Track resources before training
resources_before = get_resource_usage()
training_start_time = time.perf_counter()
print(undersampled_training_data.shape)
balanced_model, balanced_params, _, preprocessor, feature_names = finetune_oct(
    X_train=undersampled_training_data[[col for col in feature_cols]],
    y_train=undersampled_training_data[target_col],
    X_val=X_val,
    y_val=y_val,
    categorical_cols=CAT_COLUMNS,
    numeric_cols=TRUE_NUM_COLUMNS,
    depths=[ 7],
    minbuckets = [100, 150],
    cps = [0.00001, 0.0001, 0.001, 0.01]
)

training_end_time = time.perf_counter()
training_time = training_end_time - training_start_time
resources_after_training = get_resource_usage()

# Evaluate
evaluation_start_time = time.perf_counter()
metrics = evaluate_binary_oct(
    balanced_model, X_test, y_test, preprocessor, feature_names, X_val_df=X_val, y_val=y_val,
    results_dir = "msk_balanced_oct/", save_suffix=f"{balanced_params[0]}_{balanced_params[1]}_{balanced_params[2]}"
)
evaluation_end_time = time.perf_counter()
evaluation_time = evaluation_end_time - evaluation_start_time
resources_after_eval = get_resource_usage()

total_time = time.perf_counter() - training_start_time

print(f"\n{'='*80}")
print("MODEL TRAINING COMPLETE")
print(f"{'='*80}")
print(f"\n✅ Trained OCT on K-Center undersampled data:")
print(f"   - Training samples: {len(train_pd):,}")
print(f"\n⏱️  Runtime:")
print(f"   - OCT training time: {format_time(training_time)}s")
print(f"   - Evaluation time: {format_time(evaluation_time)}s")
print(f"   - Total time: {format_time(total_time)}s")
if PSUTIL_AVAILABLE:
    memory_delta_training = resources_after_training['memory_mb'] - resources_before['memory_mb']
    memory_delta_eval = resources_after_eval['memory_mb'] - resources_after_training['memory_mb']
    print(f"\n💻 Compute Resources:")
    print(f"   - Peak memory (training): {resources_after_training['memory_mb']:.1f} MB (Δ {memory_delta_training:+.1f} MB)")
    print(f"   - Peak memory (evaluation): {resources_after_eval['memory_mb']:.1f} MB (Δ {memory_delta_eval:+.1f} MB)")
    print(f"   - Peak CPU (training): {resources_after_training['cpu_percent']:.1f}%")
    print(f"   - Peak CPU (evaluation): {resources_after_eval['cpu_percent']:.1f}%")
print(f"{'='*80}\n")
balanced_model

(6792, 95)
Finetuning OCT with depths: [7], minbuckets: [100, 150], cps: [1e-05, 0.0001, 0.001, 0.01], for best PR-AUC!!!
→ Building preprocessor:
   • OneHotEncoder on: ['direct_msk_cost_pattern_2017', 'direct_msk_cost_stability_2017', 'msk_procedure_cost_pattern_2017', 'msk_procedure_cost_stability_2017', 'comorbidity_only_cost_pattern_2017', 'comorbidity_only_cost_stability_2017']
   • StandardScaler on: ['2017Q1_direct_msk_cost_3month', '2017Q1_msk_procedure_cost_3month', '2017Q1_comorbidity_only_cost_3month', '2017Q2_direct_msk_cost_3month', '2017Q2_msk_procedure_cost_3month', '2017Q2_comorbidity_only_cost_3month', '2017Q3_direct_msk_cost_3month', '2017Q3_msk_procedure_cost_3month', '2017Q3_comorbidity_only_cost_3month', '2017Q4_direct_msk_cost_3month', '2017Q4_msk_procedure_cost_3month', '2017Q4_comorbidity_only_cost_3month', 'direct_msk_cost_annual', 'msk_procedure_cost_annual', 'comorbidity_only_cost_annual', 'direct_msk_cost_deriv_Q1_Q2_2017', 'direct_msk_cost_deriv_Q2_Q3_2017